In [5]:
import pandas as pd

In [9]:
df=pd.read_excel('nadil.xlsx')

In [10]:
df.head()

,Date,Discription,Payments,Receipts,Balance,cleaned_particulars,Category,Cluster
0,2022-11-06,t ahirt OTHBNK T,6030.0,NaN,12454.64,t ahirt othbnk t,NADIL OTHBNK T,0
1,2022-11-06,010001088282101 OTHBNK T,3030.0,NaN,9424.64,010001088282101 othbnk t,OTHBNK T,1
2,2022-11-15,RIB/RMB SE.CH 20 IBMB Chg,25.0,NaN,9399.64,rib/rmb se.ch 20 ibmb charge,RIBRMB SECH IBMB CHARGE,2
3,2022-11-18,nadil Siriwardha MB SA TF,450.0,NaN,11537.14,nadil siriwardha mb sa tf,NADIL OTHBNK T,0
4,2022-12-24,nadil OTHBNK T,7530.0,NaN,27264.90,nadil othbnk t,NADIL OTHBNK T,0


In [11]:


# Assuming your Excel data is already loaded into a DataFrame 'df'
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['Payments'] = df['Payments'].astype(float)
df['Receipts'] = df['Receipts'].astype(float)
df['Balance'] = df['Balance'].astype(float)

# Create the full date range from the start to end date
start_date = df['Date'].min()
end_date = df['Date'].max()
all_dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Create a list of all clusters
all_clusters = df['Cluster'].unique()

# Create a new DataFrame to hold the final results
final_df = pd.DataFrame(index=pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster']))

# Merge the original DataFrame with the new index
df_merged = pd.merge(final_df, df, on=['Date', 'Cluster'], how='left')

# Fill missing balances by carrying the previous balance value forward
df_merged['Balance'] = df_merged['Balance'].fillna(method='ffill')

# Group by Date and Cluster, summing up Payments and Receipts where applicable
df_merged = df_merged.groupby(['Date', 'Cluster']).agg({
    'Payments': 'sum',
    'Receipts': 'sum',
    'Balance': 'last',  # Carry forward the last balance
    'cleaned_particulars': 'first',  # You can change this depending on how to handle the description
    'Category': 'first'  # Assuming you want the first category for each day/cluster
}).reset_index()

# If there are no transactions for a day, the Payments and Receipts will remain 0
df_merged['Payments'].fillna(0, inplace=True)
df_merged['Receipts'].fillna(0, inplace=True)

# Finally, fill in the balance correctly, keeping it continuous if no transactions
df_merged['Balance'] = df_merged.groupby('Cluster')['Balance'].fillna(method='ffill')

# Output the resulting DataFrame
print(df_merged)


            Date  Cluster  Payments  Receipts   Balance  \
0     2022-11-06       -1       0.0       0.0   9424.64   
1     2022-11-06        0    6030.0       0.0  12454.64   
2     2022-11-06        1    3030.0       0.0   9424.64   
3     2022-11-06        2       0.0       0.0   9424.64   
4     2022-11-06        3       0.0       0.0   9424.64   
...          ...      ...       ...       ...       ...   
10629 2025-01-31        7       0.0       0.0   1037.22   
10630 2025-01-31        8       0.0       0.0   1037.22   
10631 2025-01-31        9       0.0       0.0   1037.22   
10632 2025-01-31       10       0.0       0.0   1037.22   
10633 2025-01-31       11       0.0       0.0   1037.22   

            cleaned_particulars        Category  
0                          None            None  
1              t ahirt othbnk t  NADIL OTHBNK T  
2      010001088282101 othbnk t        OTHBNK T  
3                          None            None  
4                          None          

C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\3367905600.py:22: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_merged['Balance'] = df_merged['Balance'].fillna(method='ffill')
C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\3367905600.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_merged['Payments'].fillna(0, inplace=True)
C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\3367905600.py:35: FutureWarning: A value is trying t

In [12]:
print(df_merged.dtypes)

Date                   datetime64[ns]
Cluster                         int64
Payments                      float64
Receipts                      float64
Balance                       float64
cleaned_particulars            object
Category                       object
dtype: object


In [13]:
df_final = df_merged[['Date', 'Cluster', 'Payments','Balance']]


In [14]:
df_final = df_final.sort_values(by=['Date', 'Cluster']).reset_index(drop=True)


In [24]:
print(df_final)

            Date  Cluster  Payments   Balance
0     2022-11-06       -1       0.0   9424.64
1     2022-11-06        0    6030.0  12454.64
2     2022-11-06        1    3030.0   9424.64
3     2022-11-06        2       0.0   9424.64
4     2022-11-06        3       0.0   9424.64
...          ...      ...       ...       ...
10629 2025-01-31        7       0.0   1037.22
10630 2025-01-31        8       0.0   1037.22
10631 2025-01-31        9       0.0   1037.22
10632 2025-01-31       10       0.0   1037.22
10633 2025-01-31       11       0.0   1037.22

[10634 rows x 4 columns]


In [31]:
print(df_final.describe)

<bound method NDFrame.describe of             Date  Cluster  Payments   Balance
0     2022-11-06       -1       0.0   9424.64
1     2022-11-06        0    6030.0  12454.64
2     2022-11-06        1    3030.0   9424.64
3     2022-11-06        2       0.0   9424.64
4     2022-11-06        3       0.0   9424.64
...          ...      ...       ...       ...
10629 2025-01-31        7       0.0   1037.22
10630 2025-01-31        8       0.0   1037.22
10631 2025-01-31        9       0.0   1037.22
10632 2025-01-31       10       0.0   1037.22
10633 2025-01-31       11       0.0   1037.22

[10634 rows x 4 columns]>


In [33]:
print(df_final.describe)

<bound method NDFrame.describe of             Date  Cluster  Payments   Balance
0     2022-11-06       -1       0.0   9424.64
1     2022-11-06        0    6030.0  12454.64
2     2022-11-06        1    3030.0   9424.64
3     2022-11-06        2       0.0   9424.64
4     2022-11-06        3       0.0   9424.64
...          ...      ...       ...       ...
10629 2025-01-31        7       0.0   1037.22
10630 2025-01-31        8       0.0   1037.22
10631 2025-01-31        9       0.0   1037.22
10632 2025-01-31       10       0.0   1037.22
10633 2025-01-31       11       0.0   1037.22

[10634 rows x 4 columns]>


In [39]:
def cap_outliers_exclude_zeros(group, column='Payments'):
    non_zero = group[group[column] != 0][column]
    if non_zero.empty:
        return group

    Q1 = non_zero.quantile(0.25)
    Q3 = non_zero.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    group[column] = group[column].clip(lower=lower_bound, upper=upper_bound)
    return group

# Split the logic to avoid triggering the warning
# Step 1: group only the relevant columns
df_capped = df_final.copy()
grouped = df_capped.groupby('Cluster', group_keys=False)

# Step 2: apply the function, now warning-free
df_capped = grouped.apply(lambda g: cap_outliers_exclude_zeros(g, column='Payments')).reset_index(drop=True)


C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\1131305189.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_capped = grouped.apply(lambda g: cap_outliers_exclude_zeros(g, column='Payments')).reset_index(drop=True)


In [40]:
print(df_capped)

            Date  Cluster  Payments   Balance
0     2022-11-06       -1     6.250   9424.64
1     2022-11-06        0  6030.000  12454.64
2     2022-11-06        1  3030.000   9424.64
3     2022-11-06        2    40.000   9424.64
4     2022-11-06        3     0.000   9424.64
...          ...      ...       ...       ...
10629 2025-01-31        7     0.000   1037.22
10630 2025-01-31        8    63.125   1037.22
10631 2025-01-31        9     0.000   1037.22
10632 2025-01-31       10    50.000   1037.22
10633 2025-01-31       11  5010.000   1037.22

[10634 rows x 4 columns]


In [41]:
def standardize_zscore(group, columns=['Payments', 'Balance']):
    for col in columns:
        mean = group[col].mean()
        std = group[col].std()
        if std != 0:
            group[col + '_z'] = (group[col] - mean) / std
        else:
            group[col + '_z'] = 0  # All values same
    return group

df_standardized = (
    df_capped.groupby('Cluster', group_keys=False)
    .apply(standardize_zscore, columns=['Payments', 'Balance'])
    .reset_index(drop=True)
)


C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\820660872.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(standardize_zscore, columns=['Payments', 'Balance'])


In [42]:
print(df_standardized)

            Date  Cluster  Payments   Balance  Payments_z  Balance_z
0     2022-11-06       -1     6.250   9424.64   -0.073853  -0.404327
1     2022-11-06        0  6030.000  12454.64    7.687525  -0.274007
2     2022-11-06        1  3030.000   9424.64    9.689150  -0.405665
3     2022-11-06        2    40.000   9424.64    0.000000  -0.405763
4     2022-11-06        3     0.000   9424.64   -0.059962  -0.405988
...          ...      ...       ...       ...         ...        ...
10629 2025-01-31        7     0.000   1037.22   -0.242324  -0.761393
10630 2025-01-31        8    63.125   1037.22   -0.061666  -0.761111
10631 2025-01-31        9     0.000   1037.22   -0.336853  -0.768830
10632 2025-01-31       10    50.000   1037.22    0.000000  -0.768272
10633 2025-01-31       11  5010.000   1037.22    0.000000  -0.768287

[10634 rows x 6 columns]


In [43]:
def minmax_scale_exclude_zeros(group, columns=['Payments', 'Balance']):
    for col in columns:
        # Exclude zeros from min/max calculation
        non_zero = group[group[col] != 0][col]
        if non_zero.empty:
            group[col + '_scaled'] = group[col]
            continue

        min_val = non_zero.min()
        max_val = non_zero.max()

        # Scale: (x - min) / (max - min)
        # 0s stay 0, others get scaled
        def scale_val(x):
            if x == 0:
                return 0
            return (x - min_val) / (max_val - min_val) if max_val != min_val else 0

        group[col + '_scaled'] = group[col].apply(scale_val)
    return group

df_scaled = (
    df_capped.groupby('Cluster', group_keys=False)
    .apply(minmax_scale_exclude_zeros, columns=['Payments', 'Balance'])
).reset_index(drop=True)


C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\2090619417.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(minmax_scale_exclude_zeros, columns=['Payments', 'Balance'])


In [44]:
print(df_scaled)

            Date  Cluster  Payments   Balance  Payments_scaled  Balance_scaled
0     2022-11-06       -1     6.250   9424.64         0.000000        0.118279
1     2022-11-06        0  6030.000  12454.64         0.501348        0.159044
2     2022-11-06        1  3030.000   9424.64         0.483871        0.118279
3     2022-11-06        2    40.000   9424.64         0.000000        0.118279
4     2022-11-06        3     0.000   9424.64         0.000000        0.118279
...          ...      ...       ...       ...              ...             ...
10629 2025-01-31        7     0.000   1037.22         0.000000        0.005437
10630 2025-01-31        8    63.125   1037.22         0.000000        0.005437
10631 2025-01-31        9     0.000   1037.22         0.000000        0.005437
10632 2025-01-31       10    50.000   1037.22         0.000000        0.005437
10633 2025-01-31       11  5010.000   1037.22         0.000000        0.005437

[10634 rows x 6 columns]


In [45]:
df_final = df_scaled[['Date', 'Cluster', 'Balance_scaled', 'Payments_scaled']]


In [47]:
print(df_final.head())

        Date  Cluster  Balance_scaled  Payments_scaled
0 2022-11-06       -1        0.118279         0.000000
1 2022-11-06        0        0.159044         0.501348
2 2022-11-06        1        0.118279         0.483871
3 2022-11-06        2        0.118279         0.000000
4 2022-11-06        3        0.118279         0.000000


In [48]:
import numpy as np

df_final['day_of_week'] = df_final['Date'].dt.dayofweek
df_final['day_of_week_sin'] = np.sin(2 * np.pi * df_final['day_of_week'] / 7)
df_final['day_of_week_cos'] = np.cos(2 * np.pi * df_final['day_of_week'] / 7)


C:\Users\isuru\AppData\Local\Temp\ipykernel_20856\1761920363.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['day_of_week'] = df_final['Date'].dt.dayofweek


In [49]:
print(df_final.head())

        Date  Cluster  Balance_scaled  Payments_scaled  day_of_week  \
0 2022-11-06       -1        0.118279         0.000000            6   
1 2022-11-06        0        0.159044         0.501348            6   
2 2022-11-06        1        0.118279         0.483871            6   
3 2022-11-06        2        0.118279         0.000000            6   
4 2022-11-06        3        0.118279         0.000000            6   

   day_of_week_sin  day_of_week_cos  
0        -0.781831          0.62349  
1        -0.781831          0.62349  
2        -0.781831          0.62349  
3        -0.781831          0.62349  
4        -0.781831          0.62349  


In [50]:
df_final['month'] = df_final['Date'].dt.month
df_final['month_sin'] = np.sin(2 * np.pi * df_final['month'] / 12)
df_final['month_cos'] = np.cos(2 * np.pi * df_final['month'] / 12)


In [51]:
df_final['day'] = df_final['Date'].dt.day
df_final['month'] = df_final['Date'].dt.month
df_final['year'] = df_final['Date'].dt.year


In [52]:
df_final['7_day_rolling_payment'] = df_final.groupby('Cluster')['Payments_scaled'].rolling(window=7, min_periods=1).mean().reset_index(0, drop=True)
df_final['7_day_rolling_balance'] = df_final.groupby('Cluster')['Balance_scaled'].rolling(window=7, min_periods=1).mean().reset_index(0, drop=True)


In [53]:
non_zero_mean_payment = df_final[df_final['Payments_scaled'] != 0].groupby('Cluster')['Payments_scaled'].transform('mean')
non_zero_mean_balance = df_final[df_final['Balance_scaled'] != 0].groupby('Cluster')['Balance_scaled'].transform('mean')

df_final['payment_deviation'] = df_final['Payments_scaled'] - non_zero_mean_payment
df_final['balance_deviation'] = df_final['Balance_scaled'] - non_zero_mean_balance


In [54]:
df_final['transaction_occurred'] = (df_final['Payments_scaled'] != 0).astype(int)


In [55]:
df_final['days_since_last_transaction'] = df_final.groupby('Cluster')['Date'].transform(lambda x: x.diff().fillna(pd.Timedelta(days=0)).dt.days)


In [56]:
df_final['7_day_total_payment'] = df_final.groupby('Cluster')['Payments_scaled'].rolling(window=7, min_periods=1).sum().reset_index(0, drop=True)


In [57]:
# First, compute Recency, Frequency, and Monetary for each Category
rfm = df_final.groupby('Cluster').agg(
    recency=('Date', lambda x: (x.max() - x.min()).days),  # Recency: Days since last transaction
    frequency=('Date', 'count'),  # Frequency: Total number of transactions
    monetary=('Payments_scaled', 'sum')  # Monetary: Total amount spent
).reset_index()

# Now, we can categorize customers as 'Frequent' or 'Dormant' based on some threshold values for Recency, Frequency, and Monetary.
rfm['category_type'] = np.where(
    (rfm['recency'] <= 30) & (rfm['frequency'] >= 10) & (rfm['monetary'] >= 1000),
    'Frequent', 'Dormant'
)

# Merge back with the main DataFrame to add the category feature
df_final = df_final.merge(rfm[['Cluster', 'category_type']], on='Cluster', how='left')


In [58]:
print(df_final.dtypes)

Date                           datetime64[ns]
Cluster                                 int64
Balance_scaled                        float64
Payments_scaled                       float64
day_of_week                             int32
day_of_week_sin                       float64
day_of_week_cos                       float64
month                                   int32
month_sin                             float64
month_cos                             float64
day                                     int32
year                                    int32
7_day_rolling_payment                 float64
7_day_rolling_balance                 float64
payment_deviation                     float64
balance_deviation                     float64
transaction_occurred                    int64
days_since_last_transaction             int64
7_day_total_payment                   float64
category_type                          object
dtype: object
